# Model 1: Churn Prediction

#Importing the required pacakages

In [0]:
%restart_python 

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.formula.api as smf
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib

# Importing the data

In [0]:
VOLUME_PATH = "/Volumes/workspace/default/raw_data/"

In [0]:
demographics = pd.read_csv(VOLUME_PATH + "customer_demographics.csv")
location = pd.read_csv(VOLUME_PATH + "customer_location.csv")
services = pd.read_csv(VOLUME_PATH + "customer_services.csv")
account_status = pd.read_csv(VOLUME_PATH + "customer_account_status.csv")
zipcode_population = pd.read_csv(VOLUME_PATH + "zipcode_population.csv")

# Cleaning

In [0]:
services.head(3)

,Customer ID,Offer,Phone Service,Avg Monthly Long Distance Charges,Multiple Lines,Internet Service,Internet Type,Avg Monthly GB Download,Online Security,Online Backup,Device Protection Plan,Premium Tech Support,Streaming TV,Streaming Movies,Streaming Music,Unlimited Data
0,0002-ORFBO,NaN,Yes,42.39,No,Yes,Cable,16.0,No,Yes,No,Yes,Yes,No,No,Yes
1,0003-MKNFE,NaN,Yes,10.69,Yes,Yes,Cable,10.0,No,No,No,No,No,Yes,Yes,No
2,0004-TLHLJ,Offer E,Yes,33.65,No,Yes,Fiber Optic,30.0,No,No,Yes,No,No,No,No,Yes


In [0]:
services.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 16 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Offer                              3166 non-null   object 
 2   Phone Service                      7043 non-null   object 
 3   Avg Monthly Long Distance Charges  6361 non-null   float64
 4   Multiple Lines                     6361 non-null   object 
 5   Internet Service                   7043 non-null   object 
 6   Internet Type                      5517 non-null   object 
 7   Avg Monthly GB Download            5517 non-null   float64
 8   Online Security                    5517 non-null   object 
 9   Online Backup                      5517 non-null   object 
 10  Device Protection Plan             5517 non-null   object 
 11  Premium Tech Support               5517 non-null   objec

In [0]:
services['Internet_Type_Clean'] = services['Internet Type'].fillna('No Internet Service')
services['Offer_Clean'] = services['Offer'].fillna('No Offer')


In [0]:
def missing_value_imp(x):
  if x.dtype == 'int' or x.dtype == 'float':
    x = x.fillna(x.mean())
  else:
    x = x.fillna(x.mode()[0])
  return x

In [0]:
services = services.apply(missing_value_imp)

In [0]:
services.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 18 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Offer                              7043 non-null   object 
 2   Phone Service                      7043 non-null   object 
 3   Avg Monthly Long Distance Charges  7043 non-null   float64
 4   Multiple Lines                     7043 non-null   object 
 5   Internet Service                   7043 non-null   object 
 6   Internet Type                      7043 non-null   object 
 7   Avg Monthly GB Download            7043 non-null   float64
 8   Online Security                    7043 non-null   object 
 9   Online Backup                      7043 non-null   object 
 10  Device Protection Plan             7043 non-null   object 
 11  Premium Tech Support               7043 non-null   objec

In [0]:
account_status.head(3)

,Customer ID,Number of Referrals,Tenure in Months,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,2,9,One Year,Yes,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,0,9,Month-to-Month,No,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,0,4,Month-to-Month,Yes,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices


In [0]:
account_status.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Customer ID                  7043 non-null   object 
 1   Number of Referrals          7043 non-null   int64  
 2   Tenure in Months             7043 non-null   int64  
 3   Contract                     7043 non-null   object 
 4   Paperless Billing            7043 non-null   object 
 5   Payment Method               7043 non-null   object 
 6   Monthly Charge               7043 non-null   float64
 7   Total Charges                7043 non-null   float64
 8   Total Refunds                7043 non-null   float64
 9   Total Extra Data Charges     7043 non-null   int64  
 10  Total Long Distance Charges  7043 non-null   float64
 11  Total Revenue                7043 non-null   float64
 12  Customer Status              7043 non-null   object 
 13  Churn Category    

In [0]:
account_status['Churn_Category_Clean'] = account_status['Churn Category'].fillna('Not Churned')
account_status['Churn_Reason_Clean'] = account_status['Churn Reason'].fillna('Not Churned')

In [0]:
account_status.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Customer ID                  7043 non-null   object 
 1   Number of Referrals          7043 non-null   int64  
 2   Tenure in Months             7043 non-null   int64  
 3   Contract                     7043 non-null   object 
 4   Paperless Billing            7043 non-null   object 
 5   Payment Method               7043 non-null   object 
 6   Monthly Charge               7043 non-null   float64
 7   Total Charges                7043 non-null   float64
 8   Total Refunds                7043 non-null   float64
 9   Total Extra Data Charges     7043 non-null   int64  
 10  Total Long Distance Charges  7043 non-null   float64
 11  Total Revenue                7043 non-null   float64
 12  Customer Status              7043 non-null   object 
 13  Churn Category    

In [0]:
account_status['Has_Discount'] = np.where(account_status['Monthly Charge'] < 0, 1, 0)
account_status['Monthly_Discount_Amount'] = np.where(
    account_status['Monthly Charge'] < 0, account_status['Monthly Charge'].abs(), 0)

#Merge into one master table

In [0]:
location.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Customer ID  7043 non-null   object 
 1   City         7043 non-null   object 
 2   Zip Code     7043 non-null   int64  
 3   Latitude     7043 non-null   float64
 4   Longitude    7043 non-null   float64
dtypes: float64(2), int64(1), object(2)
memory usage: 275.2+ KB


In [0]:
data = demographics.merge(location, on='Customer ID', how='left')
data = data.merge(services, on='Customer ID', how='left')
data = data.merge(account_status, on='Customer ID', how='left')
data = data.merge(zipcode_population, on='Zip Code', how='left')

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 45 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   object 
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   object 
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Offer                              7043 non-null   object 
 10  Phone Service                      7043 non-null   object 
 11  Avg Monthly Long Distance Charges  7043 non-null   float

#Model 1: Churn Prediction

In [0]:
data.columns

Index(['Customer ID', 'Gender', 'Age', 'Married', 'Number of Dependents',
       'City', 'Zip Code', 'Latitude', 'Longitude', 'Offer', 'Phone Service',
       'Avg Monthly Long Distance Charges', 'Multiple Lines',
       'Internet Service', 'Internet Type', 'Avg Monthly GB Download',
       'Online Security', 'Online Backup', 'Device Protection Plan',
       'Premium Tech Support', 'Streaming TV', 'Streaming Movies',
       'Streaming Music', 'Unlimited Data', 'Internet_Type_Clean',
       'Offer_Clean', 'Number of Referrals', 'Tenure in Months', 'Contract',
       'Paperless Billing', 'Payment Method', 'Monthly Charge',
       'Total Charges', 'Total Refunds', 'Total Extra Data Charges',
       'Total Long Distance Charges', 'Total Revenue', 'Customer Status',
       'Churn Category', 'Churn Reason', 'Churn_Category_Clean',
       'Churn_Reason_Clean', 'Has_Discount', 'Monthly_Discount_Amount',
       'Population'],
      dtype='object')

In [0]:
data.drop(columns=['Offer','Internet Type','Churn Category','Churn Reason'], inplace=True)

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 41 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   object 
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   object 
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Phone Service                      7043 non-null   object 
 10  Avg Monthly Long Distance Charges  7043 non-null   float64
 11  Multiple Lines                     7043 non-null   objec

In [0]:
data.columns = data.columns.str.replace(' ','_')

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 41 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer_ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   object 
 4   Number_of_Dependents               7043 non-null   int64  
 5   City                               7043 non-null   object 
 6   Zip_Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Phone_Service                      7043 non-null   object 
 10  Avg_Monthly_Long_Distance_Charges  7043 non-null   float64
 11  Multiple_Lines                     7043 non-null   objec

In [0]:
data.head(3)

,Customer_ID,Gender,Age,Married,Number_of_Dependents,City,Zip_Code,Latitude,Longitude,Phone_Service,Avg_Monthly_Long_Distance_Charges,Multiple_Lines,Internet_Service,Avg_Monthly_GB_Download,Online_Security,Online_Backup,Device_Protection_Plan,Premium_Tech_Support,Streaming_TV,Streaming_Movies,Streaming_Music,Unlimited_Data,Internet_Type_Clean,Offer_Clean,Number_of_Referrals,Tenure_in_Months,Contract,Paperless_Billing,Payment_Method,Monthly_Charge,Total_Charges,Total_Refunds,Total_Extra_Data_Charges,Total_Long_Distance_Charges,Total_Revenue,Customer_Status,Churn_Category_Clean,Churn_Reason_Clean,Has_Discount,Monthly_Discount_Amount,Population
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,Yes,42.39,No,Yes,16.0,No,Yes,No,Yes,Yes,No,No,Yes,Cable,No Offer,2,9,One Year,Yes,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,Not Churned,Not Churned,0,0.0,4498
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,Yes,10.69,Yes,Yes,10.0,No,No,No,No,No,Yes,Yes,No,Cable,No Offer,0,9,Month-to-Month,No,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,Not Churned,Not Churned,1,4.0,31297
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,Yes,33.65,No,Yes,30.0,No,No,Yes,No,No,No,No,Yes,Fiber Optic,Offer E,0,4,Month-to-Month,Yes,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices,0,0.0,62069


In [0]:
data['Gender'] = pd.get_dummies(data['Gender'], drop_first=True, dtype='int')
data['Married'] = np.where(data['Married'] == 'Yes', 1, 0)
data['Phone_Service'] = np.where(data['Phone_Service'] == 'Yes', 1, 0)
data['Multiple_Lines'] = np.where(data['Multiple_Lines'] == 'Yes', 1, 0)
data['Internet_Service'] = np.where(data['Internet_Service'] == 'Yes', 1, 0)
data['Online_Security'] = np.where(data['Online_Security'] == 'Yes', 1, 0)
data['Online_Backup'] = np.where(data['Online_Backup'] == 'Yes', 1, 0)
data['Device_Protection_Plan'] = np.where(data['Device_Protection_Plan'] == 'Yes', 1, 0)
data['Premium_Tech_Support'] = np.where(data['Premium_Tech_Support'] == 'Yes', 1, 0)
data['Streaming_TV'] = np.where(data['Streaming_TV'] == 'Yes', 1, 0)
data['Streaming_Movies'] = np.where(data['Streaming_Movies'] == 'Yes', 1, 0)
data['Streaming_Music'] = np.where(data['Streaming_Music'] == 'Yes', 1, 0)
data['Unlimited_Data'] = np.where(data['Unlimited_Data'] == 'Yes', 1, 0)
data = pd.concat([data, pd.get_dummies(data['Internet_Type_Clean'], drop_first=True,dtype='int', prefix='Internet_Type_Clean')], axis=1)
data.drop('Internet_Type_Clean', axis=1, inplace=True)
data = pd.concat([data, pd.get_dummies(data['Offer_Clean'], drop_first=True, dtype='int',prefix='Offer_Clean')], axis=1)
data.drop('Offer_Clean', axis=1, inplace=True)
data = pd.concat([data, pd.get_dummies(data['Contract'], drop_first=True,dtype='int', prefix='Contract')], axis=1)
data.drop('Contract', axis=1, inplace=True)
data['Paperless_Billing'] = np.where(data['Paperless_Billing'] == 'Yes', 1, 0)
data = pd.concat([data, pd.get_dummies(data['Payment_Method'], drop_first=True,dtype='int', prefix='Payment_Method')], axis=1)
data.drop('Payment_Method', axis=1, inplace=True)
data['Customer_Status'] = np.where(data['Customer_Status'] == 'Churned', 1, 0)

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 49 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Customer_ID                              7043 non-null   object 
 1   Gender                                   7043 non-null   int64  
 2   Age                                      7043 non-null   int64  
 3   Married                                  7043 non-null   int64  
 4   Number_of_Dependents                     7043 non-null   int64  
 5   City                                     7043 non-null   object 
 6   Zip_Code                                 7043 non-null   int64  
 7   Latitude                                 7043 non-null   float64
 8   Longitude                                7043 non-null   float64
 9   Phone_Service                            7043 non-null   int64  
 10  Avg_Monthly_Long_Distance_Charges        7043 no

In [0]:
data.rename(columns={'Customer_Status': 'Churned'}, inplace=True)
data.columns

Index(['Customer_ID', 'Gender', 'Age', 'Married', 'Number_of_Dependents',
       'City', 'Zip_Code', 'Latitude', 'Longitude', 'Phone_Service',
       'Avg_Monthly_Long_Distance_Charges', 'Multiple_Lines',
       'Internet_Service', 'Avg_Monthly_GB_Download', 'Online_Security',
       'Online_Backup', 'Device_Protection_Plan', 'Premium_Tech_Support',
       'Streaming_TV', 'Streaming_Movies', 'Streaming_Music', 'Unlimited_Data',
       'Number_of_Referrals', 'Tenure_in_Months', 'Paperless_Billing',
       'Monthly_Charge', 'Total_Charges', 'Total_Refunds',
       'Total_Extra_Data_Charges', 'Total_Long_Distance_Charges',
       'Total_Revenue', 'Churned', 'Churn_Category_Clean',
       'Churn_Reason_Clean', 'Has_Discount', 'Monthly_Discount_Amount',
       'Population', 'Internet_Type_Clean_DSL',
       'Internet_Type_Clean_Fiber Optic',
       'Internet_Type_Clean_No Internet Service', 'Offer_Clean_Offer A',
       'Offer_Clean_Offer B', 'Offer_Clean_Offer C', 'Offer_Clean_Offer D',
  

In [0]:
data.columns = data.columns.str.replace(' ','_')


In [0]:
data.columns

Index(['Customer_ID', 'Gender', 'Age', 'Married', 'Number_of_Dependents',
       'City', 'Zip_Code', 'Latitude', 'Longitude', 'Phone_Service',
       'Avg_Monthly_Long_Distance_Charges', 'Multiple_Lines',
       'Internet_Service', 'Avg_Monthly_GB_Download', 'Online_Security',
       'Online_Backup', 'Device_Protection_Plan', 'Premium_Tech_Support',
       'Streaming_TV', 'Streaming_Movies', 'Streaming_Music', 'Unlimited_Data',
       'Number_of_Referrals', 'Tenure_in_Months', 'Paperless_Billing',
       'Monthly_Charge', 'Total_Charges', 'Total_Refunds',
       'Total_Extra_Data_Charges', 'Total_Long_Distance_Charges',
       'Total_Revenue', 'Churned', 'Churn_Category_Clean',
       'Churn_Reason_Clean', 'Has_Discount', 'Monthly_Discount_Amount',
       'Population', 'Internet_Type_Clean_DSL',
       'Internet_Type_Clean_Fiber_Optic',
       'Internet_Type_Clean_No_Internet_Service', 'Offer_Clean_Offer_A',
       'Offer_Clean_Offer_B', 'Offer_Clean_Offer_C', 'Offer_Clean_Offer_D',
  

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 49 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Customer_ID                              7043 non-null   object 
 1   Gender                                   7043 non-null   int64  
 2   Age                                      7043 non-null   int64  
 3   Married                                  7043 non-null   int64  
 4   Number_of_Dependents                     7043 non-null   int64  
 5   City                                     7043 non-null   object 
 6   Zip_Code                                 7043 non-null   int64  
 7   Latitude                                 7043 non-null   float64
 8   Longitude                                7043 non-null   float64
 9   Phone_Service                            7043 non-null   int64  
 10  Avg_Monthly_Long_Distance_Charges        7043 no

In [0]:
X = data.drop(['Customer_ID','Churned','City','Churn_Category_Clean', 'Churn_Reason_Clean'], axis=1)
y = data['Churned']

In [0]:
data['Churned'].value_counts()

Churned
0    5174
1    1869
Name: count, dtype: int64

# Spliting the data

In [0]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

# Feature engineering

In [0]:
vif = pd.DataFrame()
vif['feature'] = x_train.columns
vif['score'] = [ variance_inflation_factor(x_train.values, i) for i in range(len(x_train.columns))]

/local_disk0/.ephemeral_nfs/envs/pythonEnv-df491247-8fbe-4b67-907b-c4d2b77e0c8d/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/local_disk0/.ephemeral_nfs/envs/pythonEnv-df491247-8fbe-4b67-907b-c4d2b77e0c8d/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/local_disk0/.ephemeral_nfs/envs/pythonEnv-df491247-8fbe-4b67-907b-c4d2b77e0c8d/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/local_disk0/.ephemeral_nfs/envs/pythonEnv-df491247-8fbe-4b67-907b-c4d2b77e0c8d/lib/p

In [0]:
vif.loc[vif['score']<5, 'feature'].values

array(['Gender', 'Age', 'Married', 'Number_of_Dependents',
       'Phone_Service', 'Avg_Monthly_Long_Distance_Charges',
       'Multiple_Lines', 'Avg_Monthly_GB_Download', 'Online_Security',
       'Online_Backup', 'Device_Protection_Plan', 'Premium_Tech_Support',
       'Streaming_TV', 'Streaming_Music', 'Unlimited_Data',
       'Number_of_Referrals', 'Paperless_Billing',
       'Monthly_Discount_Amount', 'Population', 'Internet_Type_Clean_DSL',
       'Offer_Clean_Offer_A', 'Offer_Clean_Offer_B',
       'Offer_Clean_Offer_C', 'Offer_Clean_Offer_D',
       'Offer_Clean_Offer_E', 'Contract_One_Year', 'Contract_Two_Year',
       'Payment_Method_Credit_Card', 'Payment_Method_Mailed_Check'],
      dtype=object)

In [0]:
x = data[['Gender', 'Age', 'Married', 'Number_of_Dependents',
       'Phone_Service', 'Avg_Monthly_Long_Distance_Charges',
       'Multiple_Lines', 'Avg_Monthly_GB_Download', 'Online_Security',
       'Online_Backup', 'Device_Protection_Plan', 'Premium_Tech_Support',
       'Streaming_TV', 'Streaming_Music', 'Unlimited_Data',
       'Number_of_Referrals', 'Paperless_Billing',
       'Monthly_Discount_Amount', 'Population', 'Internet_Type_Clean_DSL',
       'Offer_Clean_Offer_A', 'Offer_Clean_Offer_B',
       'Offer_Clean_Offer_C', 'Offer_Clean_Offer_D',
       'Offer_Clean_Offer_E', 'Contract_One_Year', 'Contract_Two_Year',
       'Payment_Method_Credit_Card', 'Payment_Method_Mailed_Check']]

# Spliting the data by feature scaling

In [0]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=123)

# Checking class imbalance

In [0]:
neg, pos = np.bincount(y_train)


In [0]:
imbalance_ratio = neg / pos
imbalance_ratio

np.float64(2.7761394101876675)

# Model building

# Logist regression

In [0]:
par_grid_lr = {'C': [0.01, 0.1, 1]}

In [0]:
grid_lr = GridSearchCV(LogisticRegression(max_iter=500, class_weight='balanced'),
                        param_grid=par_grid_lr, cv=3, scoring='roc_auc', n_jobs=-1)
grid_lr = grid_lr.fit(x_train, y_train)


/databricks/python/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/databricks/python/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression

In [0]:
lr = grid_lr.best_estimator_
lr.fit(x_train, y_train)

LogisticRegression(C=1, class_weight='balanced', max_iter=500)

In [0]:
print(classification_report(y_train, lr.predict(x_train)))
print(classification_report(y_test, lr.predict(x_test)))

              precision    recall  f1-score   support

           0       0.93      0.77      0.84      4142
           1       0.57      0.85      0.68      1492

    accuracy                           0.79      5634
   macro avg       0.75      0.81      0.76      5634
weighted avg       0.84      0.79      0.80      5634

              precision    recall  f1-score   support

           0       0.92      0.75      0.83      1032
           1       0.55      0.83      0.66       377

    accuracy                           0.77      1409
   macro avg       0.74      0.79      0.75      1409
weighted avg       0.82      0.77      0.78      1409



# Random forest

In [0]:
par_grid_rf = {'n_estimators': [100], 'max_depth': [6, 10]}

In [0]:
grid_rf = GridSearchCV(RandomForestClassifier(n_jobs=-1, random_state=123, class_weight='balanced'),
                        param_grid=par_grid_rf, cv=3, scoring='roc_auc', n_jobs=-1)

In [0]:
grid_rf = grid_rf.fit(x_train, y_train)

In [0]:
rf = grid_rf.best_estimator_
rf.fit(x_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_jobs=-1,
                       random_state=123)

In [0]:
print(classification_report(y_train, rf.predict(x_train)))
print(classification_report(y_test, rf.predict(x_test)))

              precision    recall  f1-score   support

           0       0.98      0.85      0.91      4142
           1       0.70      0.96      0.81      1492

    accuracy                           0.88      5634
   macro avg       0.84      0.91      0.86      5634
weighted avg       0.91      0.88      0.89      5634

              precision    recall  f1-score   support

           0       0.91      0.81      0.86      1032
           1       0.61      0.79      0.68       377

    accuracy                           0.81      1409
   macro avg       0.76      0.80      0.77      1409
weighted avg       0.83      0.81      0.81      1409



# XGBoost

In [0]:
par_grid_xgb = {'n_estimators': [100], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}

In [0]:
grid_xgb = GridSearchCV(XGBClassifier(eval_metric='logloss', random_state=123,scale_pos_weight=imbalance_ratio),
                         param_grid=par_grid_xgb, cv=3, scoring='roc_auc', n_jobs=-1)

In [0]:
grid_xgb = grid_xgb.fit(x_train, y_train)

In [0]:
xgb = grid_xgb.best_estimator_
xgb.fit(x_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [0]:
print(classification_report(y_train, xgb.predict(x_train)))
print(classification_report(y_test, xgb.predict(x_test)))

              precision    recall  f1-score   support

           0       0.95      0.79      0.86      4142
           1       0.60      0.89      0.72      1492

    accuracy                           0.81      5634
   macro avg       0.78      0.84      0.79      5634
weighted avg       0.86      0.81      0.82      5634

              precision    recall  f1-score   support

           0       0.94      0.78      0.85      1032
           1       0.59      0.86      0.70       377

    accuracy                           0.80      1409
   macro avg       0.76      0.82      0.77      1409
weighted avg       0.84      0.80      0.81      1409



# LightGBM

In [0]:
par_grid_lgb = {'n_estimators': [100], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}

In [0]:
grid_lgb = GridSearchCV(LGBMClassifier(random_state=123, verbose=-1, class_weight='balanced'),
                         param_grid=par_grid_lgb, cv=3, scoring='roc_auc', n_jobs=-1)

In [0]:
grid_lgb = grid_lgb.fit(x_train, y_train)

In [0]:
lgbm = grid_lgb.best_estimator_
lgbm.fit(x_train, y_train)

LGBMClassifier(class_weight='balanced', max_depth=3, random_state=123,
               verbose=-1)

In [0]:
print(classification_report(y_train, lgbm.predict(x_train)))
print(classification_report(y_test, lgbm.predict(x_test)))

              precision    recall  f1-score   support

           0       0.95      0.79      0.86      4142
           1       0.60      0.88      0.72      1492

    accuracy                           0.82      5634
   macro avg       0.78      0.84      0.79      5634
weighted avg       0.86      0.82      0.82      5634

              precision    recall  f1-score   support

           0       0.93      0.77      0.85      1032
           1       0.58      0.85      0.69       377

    accuracy                           0.79      1409
   macro avg       0.76      0.81      0.77      1409
weighted avg       0.84      0.79      0.80      1409



# Comparing all the model prediction

In [0]:
score = pd.DataFrame([grid_lr.best_score_, grid_rf.best_score_, grid_xgb.best_score_, grid_lgb.best_score_])
name = pd.DataFrame(['logistic_regression', 'random_forest', 'xgboost', 'lightgbm'])
best_score = pd.concat([name, score], axis=1)
best_score.columns = ['model', 'roc_auc']
best_score.sort_values('roc_auc', ascending=False)

,model,roc_auc
2,xgboost,0.897855
3,lightgbm,0.897203
1,random_forest,0.890739
0,logistic_regression,0.880496


# Saving the best model to pkl file

In [0]:
joblib.dump(xgb, '/Volumes/workspace/default/raw_data/churn_model.pkl')

['/Volumes/workspace/default/raw_data/churn_model.pkl']

In [0]:
joblib.load('/Volumes/workspace/default/raw_data/churn_model.pkl')

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

#End